# MARICEL BUSINESS — qualification de leads

Ce notebook exécute le pipeline local et reproductible. Le scoring et l'analyse fonctionnent sans clé API ni appel réseau. L'enrichissement web/LLM est facultatif, limité et doit être activé volontairement.

## 1. Importer le pipeline

Lancez ce notebook depuis le dossier `LLM_agent_for_MARICEL_BUSINESS` après avoir installé `requirements.txt`.

In [ ]:
from pathlib import Path

import pandas as pd

from src.agent.llm_agent import generate_consultant_response
from src.tools.data_analysis_tool import analyze_leads_csv

CSV_PATH = Path('data/contacts-contacts_enriched.csv')
assert CSV_PATH.exists(), f'Fichier introuvable : {CSV_PATH}'

## 2. Analyser les leads

Par défaut, les scores existants sont recalculés selon les règles documentées. Passez `preserve_existing_score=True` si votre CRM contient des scores validés.

In [ ]:
result = analyze_leads_csv(
    CSV_PATH,
    high_value_threshold=70,
    preserve_existing_score=False,
)

result['stats'], result['model_kind'], result['model_training_rows']

## 3. Explorer les résultats

La probabilité `high_value_prob` est un signal de priorisation entraîné sur les attributs bruts. Ce n'est pas une prédiction de conversion validée.

In [ ]:
industry_performance = pd.DataFrame(result['industry_performance'])
source_performance = pd.DataFrame(result['source_performance'])
top_leads = pd.DataFrame(result['predictions'])

display(industry_performance.head(10))
display(source_performance.head(10))
display(top_leads.head(10))

## 4. Obtenir une recommandation

Sans clé API, la réponse est un diagnostic local. Avec une clé `OPENAI_API_KEY`, passez `use_llm=True` pour une reformulation IA facultative.

In [ ]:
objective = 'Identifier les leads prioritaires et les segments à activer en premier.'
print(generate_consultant_response(objective, result, use_llm=False))

## 5. Exporter le résultat

Les sorties sont écrites dans `outputs/`, un dossier exclu de Git.

In [ ]:
output_directory = Path('outputs')
output_directory.mkdir(exist_ok=True)
output_path = output_directory / 'leads_analyses.csv'
result['dataframe'].to_csv(output_path, index=False)
print(f'Export créé : {output_path}')

## Enrichissement externe (facultatif)

Utilisez-le seulement pour des sites et des données que vous êtes autorisé à traiter. L'exemple ci-dessous est volontairement commenté, limité à 25 leads et requiert `OPENAI_API_KEY` uniquement si `enrich_with_llm=True`.

```python
enriched_result = analyze_leads_csv(
    CSV_PATH,
    enrich_with_web=True,
    enrich_with_llm=False,
    enrichment_limit=25,
)
```